In [2]:
import yfinance as yf

symbols = ["AAPL", "MSFT", "SPY", "BTC-USD"]

for s in symbols:
    df = yf.download(s, start="2010-01-01")
    df.reset_index(inplace=True)
    df.to_csv(f"{s}.csv", index=False)
    print(f"Saved {s}.csv")


[*********************100%***********************]  1 of 1 completed


Saved AAPL.csv


[*********************100%***********************]  1 of 1 completed


Saved MSFT.csv


[*********************100%***********************]  1 of 1 completed


Saved SPY.csv


[*********************100%***********************]  1 of 1 completed

Saved BTC-USD.csv


LAB 1 – Dimensional Data Modeling  
Objective: Thiết kế lược đồ thân thiện với phân tích dữ liệu.  
Tasks:  
1. Create fact_trade table.  
2. Create dimension tables.  
3. Write analytical queries.


In [3]:
import pandas as pd

df = pd.read_csv("AAPL.csv")
df['Date'] = pd.to_datetime(df['Date'])

# Dimension tables
dim_time = df[['Date']].drop_duplicates()
dim_time['year'] = dim_time['Date'].dt.year
dim_time['month'] = dim_time['Date'].dt.month
dim_time['day'] = dim_time['Date'].dt.day

dim_symbol = pd.DataFrame({
    'symbol_key': [1],
    'symbol': ['AAPL'],
    'asset_type': ['Equity']
})

# Fact table
fact_trade = df[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']]
fact_trade['symbol_key'] = 1

print(fact_trade.head())


        Date                Open               High                 Low  \
0        NaT                AAPL               AAPL                AAPL   
1 2010-01-04   6.400987843132776  6.433078190699774   6.369497330115575   
2 2010-01-05   6.436077779062374  6.465768788500255  6.3955893472790475   
3 2010-01-06  6.4294788491695645  6.454971414199512   6.320611293362402   
4 2010-01-07   6.350603534853995  6.358101467426172   6.269627464769028   

               Close     Volume  symbol_key  
0               AAPL       AAPL           1  
1   6.41838264465332  493729600           1  
2  6.429479598999023  601904800           1  
3   6.32720947265625  552160000           1  
4  6.315513610839844  477131200           1  


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_2976\1297682606.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fact_trade['symbol_key'] = 1


LAB 2 – Real-Time Feature Computation  
Objective: Tính toán các đặc trưng ở chế độ truyền dữ liệu trực tuyến.  
Tasks:  
1. Maintain rolling state.  
2. Compute indicators on the fly.  
3. Emit results downstream.  

In [24]:
import pandas as pd

df['Close'] = pd.to_numeric(df['Close'], errors='coerce')
df = df.dropna(subset=['Close'])

window = 5
prices = []

for _, row in df.iterrows():
    close_price = float(row['Close'])   # ÉP KIỂU CHẮC CHẮN
    prices.append(close_price)

    if len(prices) > window:
        prices.pop(0)

    rolling_mean = sum(prices) / len(prices)
    print(row['Date'], rolling_mean)


2010-01-04 00:00:00 6.41838264465332
2010-01-05 00:00:00 6.423931121826172
2010-01-06 00:00:00 6.391690572102864
2010-01-07 00:00:00 6.372646331787109
2010-01-08 00:00:00 6.369617366790772
2010-01-11 00:00:00 6.346224403381347
2010-01-12 00:00:00 6.306276512145996
2010-01-13 00:00:00 6.3043571472167965
2010-01-14 00:00:00 6.297459125518799
2010-01-15 00:00:00 6.261169528961181
2010-01-19 00:00:00 6.2907407760620115
2010-01-20 00:00:00 6.314793395996094
2010-01-21 00:00:00 6.299318122863769
2010-01-22 00:00:00 6.229259300231933
2010-01-25 00:00:00 6.212104606628418
2010-01-26 00:00:00 6.157520771026611
2010-01-27 00:00:00 6.134427833557129
2010-01-28 00:00:00 6.081763362884521
2010-01-29 00:00:00 6.0476337432861325
2010-02-01 00:00:00 5.997608852386475
2010-02-02 00:00:00 5.9371466636657715
2010-02-03 00:00:00 5.885262203216553
2010-02-04 00:00:00 5.8418354988098145
2010-02-05 00:00:00 5.862228870391846
2010-02-08 00:00:00 5.858569812774658
2010-02-09 00:00:00 5.860549545288086
2010-02-

LAB 3 – Late Data Handling  
Objective: Đảm bảo tính chính xác đối với các sự kiện bị trì hoãn.  
Tasks:  
1. Detect late events.  
2. Recompute aggregates.  
3. Track corrections.

In [7]:
df_late = df.sample(frac=1)  # shuffle giả lập late data

df_late['event_time'] = df_late['Date']
df_late['process_time'] = pd.Timestamp.now()

late_events = df_late[df_late['event_time'] < df_late['process_time']]
print("Late events:", len(late_events))


Late events: 4042


LAB 4 – Large-Scale Backfilling  
Objective: Tải dữ liệu lịch sử hiệu quả.   
Tasks:  
1. Backfill 10+ years of data.  
2. Parallelize ingestion.  
3. Control memory usage  

In [10]:
import yfinance as yf
import pandas as pd

chunks = [
    ("1993-01-01", "2000-12-31"),
    ("2001-01-01", "2010-12-31"),
    ("2011-01-01", "2020-12-31"),
    ("2021-01-01", "2024-12-31"),
]

dfs = []

for start, end in chunks:
    df = yf.download("SPY", start=start, end=end, progress=False)
    if not df.empty:
        dfs.append(df)
    else:
        print(f"Empty chunk {start} - {end}")

spy = pd.concat(dfs)
spy.reset_index(inplace=True)

spy.to_parquet("spy_backfill.parquet")
print("Backfill SPY completed")


Backfill SPY completed


LAB 5 – Log Returns & Normalization  
Objective: Tạo các đặc trưng lợi nhuận bất biến theo tỷ lệ, phù hợp cho việc học
trên nhiều loại tài sản khác nhau.  
Tasks:  
1. Load OHLCV data and sort by symbol, timestamp.  
2. Compute simple returns and log returns.  
3. Compute rolling volatility (20-day).  
4. Normalize returns by volatility.  
5. Apply cross-sectional z-score per day.

In [14]:
import numpy as np

df['return'] = df['Close'].pct_change()
df['log_return'] = np.log(df['Close'] / df['Close'].shift(1))

df['vol_20'] = df['log_return'].rolling(20).std()
df['norm_return'] = df['log_return'] / df['vol_20']


LAB 6 – Rolling Volatility Estimators  
Objective: Ước tính độ biến động bằng cách sử dụng các công cụ ước lượng giàu
thông tin.  
Tasks:  
1. Compute close-to-close volatility.  
2. Implement Parkinson volatility using high–low prices.  
3. Implement Garman–Klass estimator.  
4. Compare estimator stability across regimes.  
5. Store rolling vol features.  

In [15]:
# Close-to-close
df['vol_close'] = df['log_return'].rolling(20).std()

# Parkinson
df['parkinson'] = (1/(4*np.log(2))) * ((np.log(df['High']/df['Low']))**2)
df['vol_parkinson'] = df['parkinson'].rolling(20).mean()

# Garman-Klass
df['gk'] = 0.5*(np.log(df['High']/df['Low'])**2) - \
           (2*np.log(2)-1)*(np.log(df['Close']/df['Open'])**2)
df['vol_gk'] = df['gk'].rolling(20).mean()


LAB 7 – Momentum Horizons  
Objective: Nắm bắt động lực trên nhiều mốc thời gian khác nhau.  
Tasks:  
1. Compute cumulative returns for 5, 20, 60 days.  
2. Apply exponential decay weighting.  
3. Compare short vs long horizon correlations.  
4. Lag all features by 1 day.  


In [16]:
df['mom_5'] = df['Close'].pct_change(5)
df['mom_20'] = df['Close'].pct_change(20)
df['mom_60'] = df['Close'].pct_change(60)

# Lag 1 day
df[['mom_5','mom_20','mom_60']] = df[['mom_5','mom_20','mom_60']].shift(1)


LAB 8 – Volume Surprise  
Objective: Phát hiện hoạt động giao dịch bất thường.  
Tasks:  
1. Compute rolling mean and std of volume.  
2. Compute volume z-score.  
3. Normalize intraday vs daily volume.  
4. Flag extreme volume events  

In [20]:
import pandas as pd

result = []

symbols = df['Volume'].columns   # LẤY SYMBOL HỢP LỆ

for symbol in symbols:
    vol = df['Volume'][symbol]

    temp = pd.DataFrame(index=vol.index)
    temp['Volume'] = vol
    temp['symbol'] = symbol

    temp['vol_mean'] = temp['Volume'].rolling(20).mean()
    temp['vol_std'] = temp['Volume'].rolling(20).std()
    temp['volume_z'] = (temp['Volume'] - temp['vol_mean']) / temp['vol_std']
    temp['extreme_volume'] = temp['volume_z'].abs() > 3

    result.append(temp)

final_df = pd.concat(result)


LAB 09 – Volatility Term Structure
Objective: Phân tích động thái rủi ro ngắn hạn so với dài hạn.
Tasks:
1. Compute short-term and long-term volatility.
2. Compute volatility ratio.
3. Detect volatility acceleration.
4. Create risk-on/off feature.

In [21]:
import pandas as pd
import numpy as np

df = pd.read_csv("AAPL.csv")
df['Date'] = pd.to_datetime(df['Date'])

for c in ['Open','High','Low','Close','Volume']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df = df.dropna().sort_values('Date').reset_index(drop=True)

df['log_return'] = np.log(df['Close'] / df['Close'].shift(1))
df['vol_short'] = df['log_return'].rolling(10).std()
df['vol_long']  = df['log_return'].rolling(60).std()
df['vol_ratio'] = df['vol_short'] / df['vol_long']
df['vol_acceleration'] = df['vol_ratio'].diff()
df['risk_regime'] = np.where(df['vol_ratio'] > 1, 'Risk-Off', 'Risk-On')

df[['vol_short','vol_long','vol_ratio','vol_acceleration','risk_regime']] = \
    df[['vol_short','vol_long','vol_ratio','vol_acceleration','risk_regime']].shift(1)

df[['Date','vol_short','vol_long','vol_ratio','risk_regime']].tail(10)


,Date,vol_short,vol_long,vol_ratio,risk_regime
4032,2026-01-14,0.007103,0.010413,0.682165,Risk-On
4033,2026-01-15,0.007063,0.010146,0.696191,Risk-On
4034,2026-01-16,0.007084,0.008863,0.799247,Risk-On
4035,2026-01-20,0.007228,0.008955,0.807110,Risk-On
4036,2026-01-21,0.011633,0.009804,1.186544,Risk-Off
4037,2026-01-22,0.011538,0.009799,1.177383,Risk-Off
4038,2026-01-23,0.011817,0.009658,1.223484,Risk-Off
4039,2026-01-26,0.011866,0.009154,1.296304,Risk-Off
4040,2026-01-27,0.015943,0.009968,1.599363,Risk-Off
4041,2026-01-28,0.016391,0.010079,1.626302,Risk-Off


LAB 10 – Calendar & Seasonality Effects
Objective: Khai thác các quy luật của lịch.
Tasks:
1. Encode day-of-week and month-end.
2. Compute average returns by bucket.
3. Normalize seasonal effects.


In [22]:
import pandas as pd
import numpy as np

df = pd.read_csv("AAPL.csv")
df['Date'] = pd.to_datetime(df['Date'])

for c in ['Close']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df = df.dropna().sort_values('Date').reset_index(drop=True)

df['return'] = df['Close'].pct_change()

df['day_of_week'] = df['Date'].dt.dayofweek
df['month_end'] = df['Date'].dt.is_month_end.astype(int)

seasonality = (
    df.groupby('day_of_week')['return']
      .mean()
      .rename('avg_return')
)

seasonality_z = (seasonality - seasonality.mean()) / seasonality.std()

pd.concat([seasonality, seasonality_z.rename('z_score')], axis=1)


,avg_return,z_score
day_of_week,,
0,0.003141,1.344251
1,0.001548,0.297516
2,0.001619,0.343979
3,-0.000547,-1.078190
4,-0.000287,-0.907556


LAB 11 – Dockerizing a Market Data ETL Pipeline
Objective: Xây dựng một pipeline ETL bằng Python để thu thập, làm sạch và lưu
trữ dữ liệu thị trường OHLCV.
Tasks
1. Create a Python ETL script:
o Read CSV market data
o Validate schema
o Write output to Parquet
2. Write a Dockerfile that:
o Uses a slim Python base image
o Installs dependencies via requirements.txt
o Copies source code into container
3. Run the container to process data using a mounted volume
4. Verify output files on the host machine

In [23]:
# --- etl.py ---
etl_code = """
import pandas as pd

df = pd.read_csv("AAPL.csv")

required_cols = ['Date','Open','High','Low','Close','Volume']
assert all(c in df.columns for c in required_cols)

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')

df.to_parquet("output/aapl.parquet", index=False)
print("ETL completed")
"""

with open("etl.py", "w") as f:
    f.write(etl_code)

# --- requirements.txt ---
with open("requirements.txt", "w") as f:
    f.write("pandas\\npyarrow\\n")

# --- Dockerfile ---
dockerfile = """
FROM python:3.10-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY etl.py .
CMD ["python", "etl.py"]
"""

with open("Dockerfile", "w") as f:
    f.write(dockerfile)

print("LAB 11 files created: etl.py, requirements.txt, Dockerfile")


LAB 11 files created: etl.py, requirements.txt, Dockerfile
